## Fraud Detection In Bitcoin Transaction Network — Improved Iteration

### What changed and why

| # | Change | Reason |
|---|--------|--------|
| 1 | **Global random seeds** | Reproducibility across runs |
| 2 | **Stratified 60/20/20 split** | Dedicated validation set; keeps class ratio |
| 3 | **Scaler fitted on training data only** | Eliminates train→test data leakage |
| 4 | **Focal Loss** | Better class-imbalance handling (illicit ~10% of labels) |
| 5 | **Deeper GAT (3 layers, 128 hidden, 8 heads, BN, Dropout)** | Higher model capacity; BN stabilises training |
| 6 | **AdamW + weight decay** | Prevents overfitting |
| 7 | **ReduceLROnPlateau + early stopping** | Avoid over-training; keep best checkpoint |
| 8 | **Threshold tuned on **validation** set** | Removes evaluation leakage |
| 9 | **Full metrics (Acc, P, R, F1, ROC-AUC, PR-AUC, CM)** | Honest assessment on imbalanced data |
| 10 | **Lightweight hyperparameter search** | Find good config without grid explosion |

In [ ]:
# ================================
# INSTALL DEPENDENCIES
# ================================
!pip install -q kagglehub torch torchvision torchaudio
!pip install -q torch-geometric
!pip install -q pandas numpy scikit-learn networkx matplotlib seaborn
!pip install -q ripser persim

In [ ]:
# ================================
# GLOBAL SEEDS — REPRODUCIBILITY
# ================================
import random, os
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", DEVICE)

In [ ]:
# ================================
# IMPORTS
# ================================
import pandas as pd
import torch.nn.functional as F
from torch import nn

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, classification_report
)
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer

import networkx as nx
from torch_geometric.nn import GATConv

import matplotlib.pyplot as plt
import seaborn as sns

from ripser import ripser

print("Imports OK")

In [ ]:
# ================================
# LOAD DATA
# ================================
import kagglehub

path = kagglehub.dataset_download("ellipticco/elliptic-data-set")
base_path = os.path.join(path, "elliptic_bitcoin_dataset")

features = pd.read_csv(f"{base_path}/elliptic_txs_features.csv", header=None)
edges    = pd.read_csv(f"{base_path}/elliptic_txs_edgelist.csv")
classes  = pd.read_csv(f"{base_path}/elliptic_txs_classes.csv")

print("Features:", features.shape)
print("Edges:   ", edges.shape)
print("Classes: ", classes.shape)

In [ ]:
# ================================
# PREPROCESS
# ================================
features.columns = ["txId", "time_step"] + [f"f{i}" for i in range(165)]

data = features.merge(classes, on="txId")

# Drop unlabelled nodes
data = data[data["class"] != "unknown"].copy()
data["class"] = data["class"].map({"1": 1, "2": 0})

time_step_vals = data["time_step"].values
X_raw = data.drop(["txId", "time_step", "class"], axis=1).values.astype(float)
y     = data["class"].values.astype(int)
tx_ids = data["txId"].values

print(f"Labelled nodes: {len(y)}  |  Illicit: {y.sum()}  |  Licit: {(1-y).sum()}")
print(f"Imbalance ratio (licit/illicit): {(1-y).sum()/y.sum():.1f}")

In [ ]:
# ================================
# GRAPH CONSTRUCTION
# ================================
valid_nodes = set(data["txId"])

edge_list = [
    (r["txId1"], r["txId2"])
    for _, r in edges.iterrows()
    if r["txId1"] in valid_nodes and r["txId2"] in valid_nodes
]

G = nx.DiGraph()
G.add_edges_from(edge_list)

node_map = {tx: i for i, tx in enumerate(tx_ids)}

edge_index_list = [
    [node_map[u], node_map[v]]
    for u, v in edge_list
]
edge_index = torch.tensor(edge_index_list, dtype=torch.long).t().contiguous()

print(f"Nodes: {len(node_map)}  |  Edges: {edge_index.shape[1]}")

In [ ]:
# ================================
# TDA FEATURES (all labelled nodes)
# ================================

def compute_tda(subgraph):
    nodes = list(subgraph.nodes())
    if len(nodes) < 3:
        return np.zeros(6)
    try:
        A    = nx.to_numpy_array(subgraph)
        dist = 1.0 - A
        np.fill_diagonal(dist, 0)
        dgms = ripser(dist, distance_matrix=True, maxdim=1)['dgms']
        h0   = dgms[0]
        h1   = dgms[1]

        h0_pers = h0[:, 1] - h0[:, 0]
        h0_pers = h0_pers[np.isfinite(h0_pers)]

        if len(h1) == 0:
            return np.array([
                len(h0_pers),
                h0_pers.max()  if len(h0_pers) else 0,
                h0_pers.mean() if len(h0_pers) else 0,
                0, 0, 0
            ])

        h1_pers = h1[:, 1] - h1[:, 0]
        return np.array([
            len(h0_pers),
            h0_pers.max()  if len(h0_pers) else 0,
            len(h1_pers),
            h1_pers.max(),
            h1_pers.mean(),
            h1_pers.sum()
        ])
    except Exception:
        return np.zeros(6)

print("Computing TDA features for ALL labelled nodes …")
tda_full = np.zeros((len(tx_ids), 6))

for tx in tx_ids:
    try:
        sub_nodes = list(
            nx.single_source_shortest_path_length(G, tx, cutoff=2).keys()
        )
        subgraph = G.subgraph(sub_nodes)
        tda_full[node_map[tx]] = compute_tda(subgraph)
    except Exception:
        pass  # leave as zeros

X = np.concatenate([X_raw, tda_full], axis=1)
print(f"Feature matrix shape after TDA: {X.shape}")

In [ ]:
# ================================
# CLEAN RAW FEATURES
# ================================
X[np.isinf(X)] = np.nan
X = np.clip(X, -1e6, 1e6)
X = np.nan_to_num(X, nan=0.0)

print("NaN count:", np.isnan(X).sum())
print("Inf count:", np.isinf(X).sum())

In [ ]:
# ================================
# TRAIN / VAL / TEST SPLIT
# Stratified 60 / 20 / 20
# Scaler fitted on TRAIN ONLY → no leakage
# ================================
idx_all = np.arange(len(y))

# 80% train+val, 20% test
idx_tv, test_idx = train_test_split(
    idx_all, test_size=0.20, random_state=SEED, stratify=y
)
# 60% train, 20% val (of original)
train_idx, val_idx = train_test_split(
    idx_tv, test_size=0.25, random_state=SEED, stratify=y[idx_tv]
)

print(f"Train: {len(train_idx)}  Val: {len(val_idx)}  Test: {len(test_idx)}")
print(f"Train illicit %: {y[train_idx].mean()*100:.1f}%")
print(f"Val   illicit %: {y[val_idx].mean()*100:.1f}%")
print(f"Test  illicit %: {y[test_idx].mean()*100:.1f}%")

# ── Fit scaler on training data ONLY ──────────────────────────────
scaler = StandardScaler()
X[train_idx] = scaler.fit_transform(X[train_idx])
X[val_idx]   = scaler.transform(X[val_idx])
X[test_idx]  = scaler.transform(X[test_idx])

X_tensor = torch.tensor(X, dtype=torch.float).to(DEVICE)
y_tensor = torch.tensor(y, dtype=torch.float).to(DEVICE)
edge_index = edge_index.to(DEVICE)

print("Tensors ready on", DEVICE)

In [ ]:
# ================================
# FOCAL LOSS
# Better than plain BCE for imbalanced data:
# down-weights easy negatives, focuses on hard positives.
# ================================

class FocalLoss(nn.Module):
    """Binary focal loss (Lin et al. 2017)."""
    def __init__(self, alpha: float = 0.25, gamma: float = 2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce   = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        p_t   = torch.sigmoid(logits) * targets + (1 - torch.sigmoid(logits)) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        loss  = alpha_t * (1 - p_t) ** self.gamma * bce
        return loss.mean()

print("FocalLoss defined")

In [ ]:
# ================================
# IMPROVED GAT MODEL
# Changes vs. baseline:
#   • 3 GAT layers (was 2)
#   • hidden_dim=128, heads=8 (was 64, 4)
#   • BatchNorm after each layer
#   • Dropout inside & between layers
#   • MLP classifier head
# ================================

class ImprovedGATNet(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, heads=8, dropout=0.4):
        super().__init__()
        self.drop = dropout

        # Layer 1: in_dim → hidden_dim * heads
        self.gat1 = GATConv(in_dim,              hidden_dim, heads=heads,
                            dropout=dropout, concat=True)
        self.bn1  = nn.BatchNorm1d(hidden_dim * heads)

        # Layer 2: hidden_dim * heads → hidden_dim * heads
        self.gat2 = GATConv(hidden_dim * heads,  hidden_dim, heads=heads,
                            dropout=dropout, concat=True)
        self.bn2  = nn.BatchNorm1d(hidden_dim * heads)

        # Layer 3: hidden_dim * heads → hidden_dim (average heads)
        self.gat3 = GATConv(hidden_dim * heads,  hidden_dim, heads=1,
                            dropout=dropout, concat=False)
        self.bn3  = nn.BatchNorm1d(hidden_dim)

        # MLP head
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.drop, training=self.training)

        x = self.gat1(x, edge_index)
        x = self.bn1(x)
        x = F.elu(x)
        x = F.dropout(x, p=self.drop, training=self.training)

        x = self.gat2(x, edge_index)
        x = self.bn2(x)
        x = F.elu(x)
        x = F.dropout(x, p=self.drop, training=self.training)

        x = self.gat3(x, edge_index)
        x = self.bn3(x)
        x = F.elu(x)
        x = F.dropout(x, p=self.drop, training=self.training)

        return self.classifier(x).squeeze(-1)   # raw logits

print("ImprovedGATNet defined")

In [ ]:
# ================================
# TRAIN IMPROVED GAT
# • AdamW + weight_decay=1e-4
# • ReduceLROnPlateau (val F1)
# • Early stopping (patience=50)
# • Gradient clipping
# ================================

def train_gat(in_dim, hidden_dim=128, heads=8, dropout=0.4,
              lr=5e-3, weight_decay=1e-4,
              focal_alpha=0.25, focal_gamma=2.0,
              max_epochs=500, patience=50,
              verbose=True):

    model = ImprovedGATNet(in_dim, hidden_dim, heads, dropout).to(DEVICE)
    criterion = FocalLoss(alpha=focal_alpha, gamma=focal_gamma)
    optimizer = torch.optim.AdamW(model.parameters(),
                                  lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=20
    )

    best_val_f1    = -1.0
    patience_count = 0
    best_state     = None

    for epoch in range(max_epochs):
        # ── Train ──────────────────────────────────────────────
        model.train()
        optimizer.zero_grad()
        logits = model(X_tensor, edge_index)
        loss   = criterion(logits[train_idx], y_tensor[train_idx])

        if torch.isnan(loss):
            print(f"NaN loss at epoch {epoch}. Stopping.")
            break

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        # ── Validate ───────────────────────────────────────────
        model.eval()
        with torch.no_grad():
            val_logits = model(X_tensor, edge_index)
            val_probs  = torch.sigmoid(val_logits).cpu().numpy()
            val_pred   = (val_probs[val_idx] > 0.5).astype(int)
            val_f1     = f1_score(y[val_idx], val_pred, zero_division=0)

        scheduler.step(val_f1)

        if val_f1 > best_val_f1:
            best_val_f1    = val_f1
            patience_count = 0
            best_state     = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_count += 1

        if patience_count >= patience:
            if verbose:
                print(f"Early stopping at epoch {epoch}  |  best val F1={best_val_f1:.4f}")
            break

        if verbose and epoch % 20 == 0:
            print(f"Epoch {epoch:4d}  loss={loss.item():.4f}  "
                  f"val_F1={val_f1:.4f}  lr={optimizer.param_groups[0]['lr']:.2e}")

    # Restore best weights
    if best_state is not None:
        model.load_state_dict(best_state)

    return model, best_val_f1


print("Starting training …")
model, best_val_f1 = train_gat(X_tensor.shape[1])
print(f"\nBest validation F1: {best_val_f1:.4f}")

In [ ]:
# ================================
# THRESHOLD TUNING — VALIDATION SET ONLY
# (never look at test labels here)
# ================================
model.eval()
with torch.no_grad():
    all_logits = model(X_tensor, edge_index)
    all_probs  = torch.sigmoid(all_logits).cpu().numpy()

thresholds  = np.arange(0.05, 0.95, 0.01)
best_thresh = 0.5
best_f1_val = 0.0

for t in thresholds:
    pred_v = (all_probs[val_idx] > t).astype(int)
    f1_v   = f1_score(y[val_idx], pred_v, zero_division=0)
    if f1_v > best_f1_val:
        best_f1_val = f1_v
        best_thresh = t

print(f"Best threshold (val):  {best_thresh:.2f}")
print(f"Val F1 at best thresh: {best_f1_val:.4f}")

In [ ]:
# ================================
# FINAL EVALUATION — TEST SET
# ================================
pred_test = (all_probs[test_idx] > best_thresh).astype(int)

print("=" * 55)
print("  IMPROVED TDA + GAT  —  TEST SET RESULTS")
print("=" * 55)
print(f"  Threshold : {best_thresh:.2f}")
print(f"  Accuracy  : {accuracy_score(y[test_idx], pred_test):.4f}")
print(f"  Precision : {precision_score(y[test_idx], pred_test, zero_division=0):.4f}")
print(f"  Recall    : {recall_score(y[test_idx], pred_test, zero_division=0):.4f}")
print(f"  F1        : {f1_score(y[test_idx], pred_test, zero_division=0):.4f}")
print(f"  ROC-AUC   : {roc_auc_score(y[test_idx], all_probs[test_idx]):.4f}")
print(f"  PR-AUC    : {average_precision_score(y[test_idx], all_probs[test_idx]):.4f}")
print("=" * 55)
print()
print(classification_report(
    y[test_idx], pred_test,
    target_names=["Licit (0)", "Illicit (1)"],
    zero_division=0
))

# ── Confusion matrix ───────────────────────────────────────────────────
cm = confusion_matrix(y[test_idx], pred_test)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Licit', 'Illicit'],
            yticklabels=['Licit', 'Illicit'])
plt.title('Confusion Matrix — Improved GAT (TDA)')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

In [ ]:
# ================================
# ROC & PR CURVES
# ================================
from sklearn.metrics import roc_curve, precision_recall_curve

fpr, tpr, _  = roc_curve(y[test_idx], all_probs[test_idx])
prec, rec, _ = precision_recall_curve(y[test_idx], all_probs[test_idx])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ROC
axes[0].plot(fpr, tpr, label=f"ROC-AUC = {roc_auc_score(y[test_idx], all_probs[test_idx]):.4f}")
axes[0].plot([0, 1], [0, 1], 'k--')
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curve")
axes[0].legend()

# PR
axes[1].plot(rec, prec, label=f"PR-AUC = {average_precision_score(y[test_idx], all_probs[test_idx]):.4f}")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ================================
# LIGHTWEIGHT HYPERPARAMETER SEARCH
# Grid is small to stay practical;
# extend as compute budget allows.
# ================================

hp_grid = [
    dict(hidden_dim=128, heads=8,  dropout=0.4, lr=5e-3),
    dict(hidden_dim=128, heads=8,  dropout=0.3, lr=3e-3),
    dict(hidden_dim=256, heads=4,  dropout=0.4, lr=5e-3),
    dict(hidden_dim=128, heads=4,  dropout=0.5, lr=1e-3),
]

results = []
for cfg in hp_grid:
    print(f"\nConfig: {cfg}")
    m, vf1 = train_gat(
        X_tensor.shape[1],
        hidden_dim   = cfg['hidden_dim'],
        heads        = cfg['heads'],
        dropout      = cfg['dropout'],
        lr           = cfg['lr'],
        verbose      = False,
        max_epochs   = 300,
        patience     = 40,
    )
    # Quick val-set threshold search
    m.eval()
    with torch.no_grad():
        p = torch.sigmoid(m(X_tensor, edge_index)).cpu().numpy()
    best_t, best_f = 0.5, 0.0
    for t in np.arange(0.05, 0.95, 0.02):
        f = f1_score(y[val_idx], (p[val_idx] > t).astype(int), zero_division=0)
        if f > best_f:
            best_f, best_t = f, t
    results.append({**cfg, 'val_f1': best_f, 'best_thresh': best_t})
    print(f"  → val F1 = {best_f:.4f}  (thresh={best_t:.2f})")

# ── Show ranked results ────────────────────────────────────────────────
print("\n" + "=" * 60)
print("HYPERPARAMETER SEARCH SUMMARY (sorted by val F1)")
print("=" * 60)
for r in sorted(results, key=lambda x: x['val_f1'], reverse=True):
    print(r)

In [ ]:
# ================================
# BASELINE COMPARISON
# Random Forest & SVM on raw features
# ================================

# Use the (scaler-transformed) X arrays for fair comparison
X_train_np = X[train_idx]
X_val_np   = X[val_idx]
X_test_np  = X[test_idx]

# Impute any residual NaNs (safety net)
imp = SimpleImputer(strategy='mean')
X_train_np = imp.fit_transform(X_train_np)
X_val_np   = imp.transform(X_val_np)
X_test_np  = imp.transform(X_test_np)

# ── Random Forest ──────────────────────────────────────────────────────
print("Training Random Forest …")
rf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=SEED,
                            class_weight='balanced')
rf.fit(X_train_np, y[train_idx])
pred_rf = rf.predict(X_test_np)
print("\nRandom Forest — Test Set:")
print(f"  Accuracy : {accuracy_score(y[test_idx], pred_rf):.4f}")
print(f"  F1       : {f1_score(y[test_idx], pred_rf, zero_division=0):.4f}")
print(f"  ROC-AUC  : {roc_auc_score(y[test_idx], rf.predict_proba(X_test_np)[:,1]):.4f}")

# ── SVM ────────────────────────────────────────────────────────────────
print("\nTraining SVM …")
svm = SVC(class_weight='balanced', probability=True, random_state=SEED)
svm.fit(X_train_np, y[train_idx])
pred_svm = svm.predict(X_test_np)
print("\nSVM — Test Set:")
print(f"  Accuracy : {accuracy_score(y[test_idx], pred_svm):.4f}")
print(f"  F1       : {f1_score(y[test_idx], pred_svm, zero_division=0):.4f}")
print(f"  ROC-AUC  : {roc_auc_score(y[test_idx], svm.predict_proba(X_test_np)[:,1]):.4f}")

## Results Summary

| Model | Accuracy | F1 | ROC-AUC |
|-------|----------|-----|---------|
| Baseline GAT + TDA (prior) | ~96.5% | ~? | ~? |
| **Improved GAT + TDA (this run)** | *see above* | *see above* | *see above* |
| Random Forest (baseline) | *see above* | *see above* | *see above* |
| SVM (baseline) | *see above* | *see above* | *see above* |

### Key improvements applied
1. **Reproducible** — fixed global SEED=42
2. **Honest evaluation** — 3-way stratified split; scaler/threshold fitted on train/val only
3. **Better model** — 3-layer GAT, BN, wider hidden dim
4. **Focal loss** — handles 10:1 class imbalance
5. **LR scheduling + early stopping** — avoids overfitting
6. **Val-set threshold tuning** — maximises F1 without peeking at test

### If >99% accuracy is not reached
The Elliptic dataset is naturally hard due to:
- Low illicit prevalence (~10 %)
- High variation in node connectivity
- Temporal structure

Next steps if still below target:
- Graph-level temporal features (illicit rate per time-step)
- GraphSAGE or GIN layers for inductive generalisation
- Ensemble of GAT + Random Forest predictions
- SMOTE or graph-aware oversampling